# E2 Stage-1 selection (Kaggle) — one notebook, MODE switch

**Accelerator: `GPU T4 x1`** (never P100 / T4x2). Attach your uploaded dataset, then set `REPO` below.

Run one phase at a time by setting `MODE` in the config cell:

| MODE | how to run | what it does |
|---|---|---|
| `score_qwen` | interactive (Run All) | score the pre-trained qwen adapter (in the dataset) on all splits — minutes |
| `train` | **Save Version -> Save & Run All (Commit)** | train `MODEL` (llama/granite), adapter only, ~8h |
| `score_adapter` | interactive | score a freshly-trained adapter (`ADAPTER`) on all splits |

After each phase, download `/kaggle/working/*.zip` and run `rank_stage1.py` locally.

In [ ]:
# ---- deps (PINNED — do not use `-U` latest) ----
# Kaggle now ships transformers 5.0.0, but the latest trl (1.10.0) is a
# transformers-4.x-era release: it imports `is_torch_distributed_available`
# from transformers.utils, which 5.0.0 removed -> ImportError, and its
# SFTConfig inheritance breaks (the `warmup_ratio` TypeError). Pin transformers
# to the last 4.x so trl 1.10.0's SFTConfig(max_length=, loss_type=, warmup_ratio=)
# all resolve. Internet must be ON (Settings -> Internet).
# NOTE: this DOWNGRADES the preinstalled transformers/datasets. In an
# interactive session do Run -> Restart & Run All after this cell so the old
# transformers isn't left imported. A fresh Commit run needs no restart.
!pip install -q "trl==1.10.0" "transformers>=4.56,<5" "peft>=0.17,<0.20" \
    "accelerate>=1.6,<1.14" "datasets>=3,<5" "bitsandbytes>=0.46.1"

In [ ]:
# ================== CONFIG — edit these ==================
MODE  = "score_qwen"        # "score_qwen" | "train" | "score_adapter"
MODEL = "llama3.2-1b"       # used by train / score_adapter: llama3.2-1b | granite-guardian-2b

# Path to the uploaded dataset (Data panel -> copy path). Must end in /cascade-pid
REPO  = "/kaggle/input/REPLACE-WITH-YOUR-SLUG/cascade-pid"

# For MODE=='score_adapter' only: path to the trained adapter from a prior
# training commit's OUTPUT (add that output as an input dataset first).
ADAPTER = f"/kaggle/input/REPLACE-WITH-TRAINING-OUTPUT/results/stage1/{MODEL}/adapter"

MAX_SAMPLES = 12000        # training subset (keep identical across candidates)
# ========================================================

In [ ]:
# ---- setup (run for every MODE) ----
import os, sys, subprocess, shutil, glob
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # single-GPU pin (kills T4x2 sharding)

# auto-detect REPO (overrides the CONFIG placeholder; robust to dataset slug/nesting)
_hits = glob.glob("/kaggle/input/**/src/models/prompt_template.py", recursive=True)
assert _hits, "dataset not attached? Right panel -> Add Input -> your dataset"
REPO = _hits[0].split("/src/")[0]
print("REPO =", REPO)
sys.path.insert(0, REPO)

import torch
print("torch sees GPUs:", torch.cuda.device_count(), "(want 1)")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

# Gated models (llama/granite) need an HF token. On Kaggle, secrets are NOT in
# os.environ -- read via UserSecretsClient, and the secret must be ATTACHED to
# this notebook: Add-ons -> Secrets -> add HF_TOKEN -> toggle it on. Also accept
# the model license on that HF account. (qwen is public -- needs no token.)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login
    login(os.environ["HF_TOKEN"]); print("HF login OK")
except Exception as e:
    print("HF login skipped (fine for qwen):", e)

In [ ]:
# ---- shared scoring helper (payload-safe: never prints text) ----
from src.models.prompt_template import load_model_config
from src.models.stage1 import Stage1Detector
from src.utils.io import read_jsonl, write_jsonl

SPLITS = [
    "data/train_proposal/val.jsonl",
    "data/train_proposal/cal.jsonl",
    "data/splits/test_in_dist.jsonl",
    "data/splits/test_cross_channel.jsonl",
    "data/splits/test_cross_domain.jsonl",
]

def score(model_name, adapter, out_dir):
    cfg = load_model_config(f"{REPO}/configs/models/{model_name}.yaml")
    det = Stage1Detector(cfg, adapter_path=adapter, device="cuda", load_in_4bit=True)
    p = det.p_safe("Please summarize the meeting notes for tomorrow.")
    assert 0.0 <= p <= 1.0; print(f"adapter OK (p_safe sample={p:.3f})")
    os.makedirs(out_dir, exist_ok=True)
    for s in SPLITS:
        path = f"{REPO}/{s}"
        if not os.path.exists(path):
            print("skip missing", s); continue
        texts = [(r.get("input") or r.get("text") or r.get("rendered_input") or "") for r in read_jsonl(path)]  # not printed
        n_empty = sum(1 for t in texts if not t)
        assert n_empty == 0, f"{s}: {n_empty}/{len(texts)} rows have empty text \u2014 check payload field name (input/text/rendered_input)"
        stem = os.path.basename(s).replace(".jsonl", "")
        write_jsonl(f"{out_dir}/{stem}_logits.jsonl", det.predict(texts))
        print(f"  scored {stem}: {len(texts)} rows")

def zip_out(name, folder):
    z = shutil.make_archive(f"/kaggle/working/{name}", "zip", folder)
    print("zipped ->", z); return z

In [ ]:
# ---- dispatch on MODE (a commit runs exactly one branch) ----
if MODE == "score_qwen":
    out = "/kaggle/working/results/stage1/qwen2.5-1.5b"
    score("qwen2.5-1.5b", f"{REPO}/results/stage1/qwen2.5-1.5b/adapter", out)
    zip_out("qwen2.5-1.5b_logits", out)

elif MODE == "train":
    out = f"/kaggle/working/results/stage1/{MODEL}"
    cmd = [
        "python", f"{REPO}/scripts/train_stage1.py",
        "--config",   f"{REPO}/configs/models/{MODEL}.yaml",
        "--train-file", f"{REPO}/data/train_proposal/train.jsonl",
        "--training", f"{REPO}/configs/training_kaggle.yaml",
        "--output-dir", out,
        "--max-samples", str(MAX_SAMPLES),
        "--no-dump-logits",          # <-- adapter only; no timeout from logit dump
    ]
    print(" ".join(cmd))
    rc = subprocess.run(cmd).returncode
    assert rc == 0, f"training failed rc={rc}"
    zip_out(f"{MODEL}_adapter", out)   # download this, add as input for score_adapter

elif MODE == "score_adapter":
    out = f"/kaggle/working/results/stage1/{MODEL}"
    score(MODEL, ADAPTER, out)
    zip_out(f"{MODEL}_logits", out)

else:
    raise ValueError(f"unknown MODE {MODE!r}")

print("\nDONE. Download /kaggle/working/*.zip from the Output/Data panel.")